Stage 1: Ingest application

Receive applicant record from dataset.

Stage 2: Validate and prepare

Clean fields, encode variables, create model-ready representation.

Stage 3: Score risk

Generate predicted default probability or risk score.

Stage 4: Apply decision policy

Convert score into approve / reject / review.

Stage 5: Generate explanation

Summarize why the recommendation was made.

Stage 6: Log result

Store outputs for evaluation and future analysis.

That is the core MVP architecture

Future extension 1: Policy service

A separate rule module for bank policy constraints.

Future extension 2: Human review queue

Cases routed to analyst review.

Future extension 3: Agent layer with Google ADK

Agent can:

summarize case

retrieve policy rules

explain borderline decisions

generate analyst notes

Future extension 4: Monitoring

Track approval rates, defaults, and threshold drift.

So this architecture is not just MVP-friendly.
It is growth-friendly.

In [1]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

df = pd.read_csv(url, sep=" ", header=None)

df.head()

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [13]:
columns = [
    "checking_account_status",
    "duration_months",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_length",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_since",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "num_dependents",
    "telephone",
    "foreign_worker",
    "target"
]

df.columns = columns

df.head()

,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_length,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   checking_account_status  1000 non-null   object
 1   duration_months          1000 non-null   int64 
 2   credit_history           1000 non-null   object
 3   purpose                  1000 non-null   object
 4   credit_amount            1000 non-null   int64 
 5   savings_account          1000 non-null   object
 6   employment_length        1000 non-null   object
 7   installment_rate         1000 non-null   int64 
 8   personal_status_sex      1000 non-null   object
 9   other_debtors            1000 non-null   object
 10  residence_since          1000 non-null   int64 
 11  property                 1000 non-null   object
 12  age                      1000 non-null   int64 
 13  other_installment_plans  1000 non-null   object
 14  housing                  1000 non-null   

In [4]:
df.nunique()

,0
checking_account_status,4
duration_months,33
credit_history,5
purpose,10
credit_amount,921
savings_account,5
employment_length,5
installment_rate,4
personal_status_sex,4
other_debtors,3


In [7]:
df.describe()

,duration_months,credit_amount,installment_rate,residence_since,age,existing_credits,num_dependents,target
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,1.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,2.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,2.000000


In [8]:
df["target"].value_counts()

,count
target,
1,700
2,300


In [12]:
df.select_dtypes(include = 'object').columns

Index(['checking_account_status', 'credit_history', 'purpose',
       'savings_account', 'employment_length', 'personal_status_sex',
       'other_debtors', 'property', 'other_installment_plans', 'housing',
       'job', 'telephone', 'foreign_worker'],
      dtype='object')

In [10]:
df.select_dtypes(include="int64").columns

Index(['duration_months', 'credit_amount', 'installment_rate',
       'residence_since', 'age', 'existing_credits', 'num_dependents',
       'target'],
      dtype='object')

In [14]:
df.duplicated().sum()

np.int64(0)

In [22]:
(df.nunique() == 2).sum()

np.int64(4)

In [23]:
df.columns[df.nunique() == 2]

Index(['num_dependents', 'telephone', 'foreign_worker', 'target'], dtype='object')